# Single Response Testing

This notebook tests the complete question answering pipeline:
1. Question handler for different types
2. LLM response generation
3. SSR rating
4. Response formatting

In [1]:
import sys
from pathlib import Path
import os
import json
from dotenv import load_dotenv

sys.path.append(str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / '.env')

from src.persona.generator import PersonaGenerator
from src.llm.client import LLMClient
from src.llm.prompts import PromptBuilder
from src.ssr.embeddings import EmbeddingService
from src.ssr.rating_engine import RatingEngine
from src.scales.registry import create_default_scales
from src.survey.question_handler import QuestionHandler, QUESTION_METADATA

import warnings
warnings.filterwarnings('ignore')

## 1. Initialize Components

In [2]:
# Initialize
persona_gen = PersonaGenerator(random_seed=42)
llm_client = LLMClient(temperature=0.5)
embedding_service = EmbeddingService()
scale_registry = create_default_scales()
rating_engine = RatingEngine(scale_registry, embedding_service)
question_handler = QuestionHandler(llm_client, rating_engine)

print("✓ All components initialized")

✓ All components initialized


## 2. Load Test Concept

In [3]:
# Load concepts from parsed data
concepts_file = Path.cwd().parent / 'data/parsed/concepts.json'

if concepts_file.exists():
    with open(concepts_file, 'r') as f:
        concepts = json.load(f)
    print(f"✓ Loaded {len(concepts)} concepts")
    test_concept = concepts[0]
else:
    # Fallback concept
    test_concept = {
        'id': 'Concept 1',
        'name': 'Christmas AR Message',
        'description': '''Christmas AR Message Scratchcard
Price: 250 CZK
- Record personal Christmas video with festive AR filters
- Recipient scans QR code to see AR message above card
- Available in stores and online''',
        'price': '250 CZK'
    }

print(f"\nTest Concept: {test_concept['name']}")
print(test_concept['description'])

✓ Loaded 8 concepts

Test Concept: Concept 1: Christmas AR Message
Concept 1: Christmas AR Message
Price: Imagine that a new personalized video scratch card has just been launched!

🎁 This Christmas gift scratch card is sold for 250 CZK and guarantees a win for the recipient — making it an ideal gift under the Christmas tree.

On the front side of the card, there is a QR code that can be scanned with a phone. After scanning, you can record a short video where you wish the recipient a Merry Christmas. A festive filter with a holiday design 🎄 is automatically added to the video.

When the recipient scratches the card, discovers their prize, and scans the QR code, your personal greeting will appear directly above the card — brought to life through augmented reality and fun digital effects.

Available at all sales points where scratch cards are sold, or online through relevant websites and apps.
Occasion: Christmas


## 3. Generate Test Persona

In [4]:
persona = persona_gen.generate_persona()
print(f"Test Persona: {persona.to_description()}")

Test Persona: 26 years old, female, works in Finance, buys lottery played in-store/person, paper scratchcards bought in person, lottery played online/app, likes to try new and different products


## 4. Test Different Question Types

In [5]:
# Test questions to handle
test_questions = ['B2', 'B3', 'B6', 'B12', 'B13', 'B7', 'B15']

results = {}

for q_id in test_questions:
    print(f"\n{'='*80}")
    print(f"Testing Question: {q_id}")
    print(f"{'='*80}")
    
    # Get metadata
    meta = QUESTION_METADATA.get(q_id, {})
    print(f"Type: {meta.get('type', 'unknown')}")
    print(f"Scale: {meta.get('scale_id', 'N/A')}")
    
    # Handle question
    try:
        response = question_handler.handle_question(
            question_id=q_id,
            question_type=meta.get('type', 'open_text'),
            persona=persona,
            concept_description=test_concept['description'],
            scale_id=meta.get('scale_id'),
            price=test_concept.get('price', '')
        )
        
        print(f"\nLLM Response:\n{response.raw_response}")
        print(f"\nFormatted: {response.formatted_response}")
        
        if response.rating_result:
            print(f"Confidence: {response.metadata.get('confidence', 0):.3f}")
            
        results[q_id] = response
        
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()


Testing Question: B2
Type: single_coded
Scale: likert5_purchase_intent_v1

LLM Response:
I find the concept quite intriguing, especially with the personalized video feature and AR effects, which add a unique touch to a traditional gift. However, the price of 250 CZK seems a bit steep for a scratchcard, even with guaranteed winnings. I might consider buying it as a special gift for someone close, but I'd have to weigh it against other gift options at that price point.

Formatted: (2) Probably would
Confidence: 0.209

Testing Question: B3
Type: single_coded
Scale: likert5_uniqueness_v1

LLM Response:
This concept feels quite new and different compared to traditional scratchcards. The integration of a personalized video message with augmented reality adds a unique, interactive twist that I haven't seen before. It combines the excitement of winning with a personal touch, making it a thoughtful gift option for the holidays.

Formatted: (2) Very new and different
Confidence: 0.301

Testing 

## 5. Response Summary

In [6]:
# Create summary table
import pandas as pd

summary_data = []
for q_id, response in results.items():
    summary_data.append({
        'Question': q_id,
        'Type': response.question_type,
        'Formatted Response': response.formatted_response,
        'Confidence': response.metadata.get('confidence', 'N/A')
    })

df_summary = pd.DataFrame(summary_data)
print("\nResponse Summary:")
print(df_summary.to_string())


Response Summary:
  Question       Type                                                                                                                                                                                                                                                                                                                                                       Formatted Response Confidence
0       B2     likert                                                                                                                                                                                                                                                                                                                                                       (2) Probably would   0.208813
1       B3     likert                                                                                                                                                                            

## 6. Test Response Consistency

In [7]:
# Test same question multiple times
print("Testing response consistency for B2 (Purchase Intent)")
print("="*80)

n_tests = 5
consistency_results = []

for i in range(n_tests):
    response = question_handler.handle_question(
        question_id='B2',
        question_type='single_coded',
        persona=persona,
        concept_description=test_concept['description'],
        scale_id='likert5_purchase_intent_v1'
    )
    
    consistency_results.append({
        'Run': i+1,
        'Response': response.formatted_response,
        'Level': response.rating_result.chosen_level_value,
        'Confidence': response.metadata.get('confidence')
    })

df_consistency = pd.DataFrame(consistency_results)
print(df_consistency)

print(f"\nMode Rating: {df_consistency['Level'].mode()[0]}")
print(f"Mean Confidence: {df_consistency['Confidence'].mean():.3f}")

Testing response consistency for B2 (Purchase Intent)
   Run              Response  Level  Confidence
0    1  (1) Definitely would      1    0.217023
1    2    (2) Probably would      2    0.210363
2    3    (2) Probably would      2    0.206867
3    4  (1) Definitely would      1    0.212040
4    5    (2) Probably would      2    0.211692

Mode Rating: 2
Mean Confidence: 0.212


## Summary

Question handling tested successfully:

✅ **Likert Questions** - SSR properly maps free-text to scale levels

✅ **Binary Questions** - Yes/No handled correctly

✅ **Slider Questions** - 9-point scale rating working

✅ **Open Text** - Direct LLM responses captured

✅ **Consistency** - Multiple runs show reasonable consistency with natural variance

**Next Steps:**
- Test conditional logic (notebook 05)
- Verify skip patterns work correctly